<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/04_Complete_GPT_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell #1: Setup and Model Configuration
# Day 6-7: Building a Complete GPT-Style Model

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("\n" + "="*60)
print("GPT MODEL CONFIGURATION")
print("="*60)

# Model hyperparameters
class GPTConfig:
    """Configuration for our GPT model"""
    vocab_size = 50257      # GPT-2 vocabulary size
    max_seq_len = 256       # Maximum sequence length
    n_layers = 6            # Number of transformer blocks
    d_model = 512           # Embedding dimension
    num_heads = 8           # Number of attention heads
    d_ff = 2048             # Feed-forward hidden dimension
    dropout = 0.1           # Dropout rate

config = GPTConfig()

print(f"\nVocabulary Size: {config.vocab_size:,}")
print(f"Max Sequence Length: {config.max_seq_len}")
print(f"Number of Layers: {config.n_layers}")
print(f"Model Dimension (d_model): {config.d_model}")
print(f"Attention Heads: {config.num_heads}")
print(f"Feed-Forward Dimension: {config.d_ff}")
print(f"Dropout: {config.dropout}")

print("\n✅ Configuration ready!")

Using device: cuda
GPU: Tesla T4
Memory Available: 15.83 GB

GPT MODEL CONFIGURATION

Vocabulary Size: 50,257
Max Sequence Length: 256
Number of Layers: 6
Model Dimension (d_model): 512
Attention Heads: 8
Feed-Forward Dimension: 2048
Dropout: 0.1

✅ Configuration ready!


In [ ]:
# Cell #2: Core Transformer Components (from previous notebooks)

class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        # Linear projections and reshape to [batch, num_heads, seq_len, d_k]
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Apply mask if provided (for causal attention)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # Apply attention to values
        output = torch.matmul(attention_weights, V)

        # Reshape and apply output projection
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.W_o(output)

        return output

class FeedForward(nn.Module):
    """Feed-forward network"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

class LayerNorm(nn.Module):
    """Layer normalization"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

class TransformerBlock(nn.Module):
    """Complete transformer block"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: Multi-head attention with residual connection
        attn_output = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        # Sub-layer 2: Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

print("✅ Core components ready!")
print("  • MultiHeadAttention")
print("  • FeedForward")
print("  • LayerNorm")
print("  • TransformerBlock")

✅ Core components ready!
  • MultiHeadAttention
  • FeedForward
  • LayerNorm
  • TransformerBlock


In [ ]:
# Cell #3: Positional Encoding (Learned Embeddings - GPT style)

class PositionalEmbedding(nn.Module):
    """Learned positional embeddings"""
    def __init__(self, max_seq_len, d_model, dropout=0.1):
        super().__init__()
        self.position_embeddings = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x: Token embeddings [batch_size, seq_len, d_model]
        Returns:
            x + positional embeddings [batch_size, seq_len, d_model]
        """
        batch_size, seq_len, d_model = x.shape

        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(0, seq_len, device=x.device)

        # Get position embeddings and add to input
        pos_emb = self.position_embeddings(positions)
        x = x + pos_emb

        return self.dropout(x)

# Test it
pos_emb = PositionalEmbedding(config.max_seq_len, config.d_model, config.dropout)
print("✅ Positional Embedding ready!")
print(f"   Max sequence length: {config.max_seq_len}")
print(f"   Embedding dimension: {config.d_model}")
print(f"   Parameters: {config.max_seq_len * config.d_model:,}")

✅ Positional Embedding ready!
   Max sequence length: 256
   Embedding dimension: 512
   Parameters: 131,072


In [ ]:
# Cell #4: Causal Mask (for autoregressive attention)

def create_causal_mask(seq_len, device):
    """
    Creates a causal mask to prevent attention to future positions.

    For seq_len=4, the mask looks like:
    [[1, 0, 0, 0],   <- token 0 can only see token 0
     [1, 1, 0, 0],   <- token 1 can see tokens 0,1
     [1, 1, 1, 0],   <- token 2 can see tokens 0,1,2
     [1, 1, 1, 1]]   <- token 3 can see all tokens

    Args:
        seq_len: Sequence length
        device: Device to create tensor on
    Returns:
        Causal mask [1, 1, seq_len, seq_len]
    """
    # Create lower triangular matrix
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Add batch and head dimensions: [1, 1, seq_len, seq_len]
    mask = mask.unsqueeze(0).unsqueeze(0)
    return mask

# Test it
test_mask = create_causal_mask(5, device)
print("✅ Causal mask function ready!")
print(f"\nMask shape: {test_mask.shape}")
print(f"\nExample causal mask (seq_len=5):")
print(test_mask.squeeze().int())
print("\n💡 Each token can only attend to itself and previous tokens!")

✅ Causal mask function ready!

Mask shape: torch.Size([1, 1, 5, 5])

Example causal mask (seq_len=5):
tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], device='cuda:0', dtype=torch.int32)

💡 Each token can only attend to itself and previous tokens!


In [ ]:
# Cell #5: Complete GPT Model

class GPT(nn.Module):
    """Complete GPT-style language model"""
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embeddings: [vocab_size] -> [d_model]
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)

        # Positional embeddings
        self.positional_embedding = PositionalEmbedding(
            config.max_seq_len,
            config.d_model,
            config.dropout
        )

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config.d_model,
                config.num_heads,
                config.d_ff,
                config.dropout
            ) for _ in range(config.n_layers)
        ])

        # Final layer normalization
        self.ln_f = LayerNorm(config.d_model)

        # Language modeling head (projects to vocabulary)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: share weights between token embeddings and lm_head
        self.lm_head.weight = self.token_embedding.weight

        print(f"✅ GPT Model initialized!")

    def forward(self, idx):
        """
        Args:
            idx: Token indices [batch_size, seq_len]
        Returns:
            logits: [batch_size, seq_len, vocab_size]
        """
        batch_size, seq_len = idx.shape

        # Token embeddings: [batch, seq_len] -> [batch, seq_len, d_model]
        token_emb = self.token_embedding(idx)

        # Add positional embeddings
        x = self.positional_embedding(token_emb)

        # Create causal mask
        mask = create_causal_mask(seq_len, idx.device)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x, mask)

        # Final layer norm
        x = self.ln_f(x)

        # Project to vocabulary: [batch, seq_len, d_model] -> [batch, seq_len, vocab_size]
        logits = self.lm_head(x)

        return logits

# Create the model
model = GPT(config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1e6:.1f} MB (fp32)")

✅ GPT Model initialized!

📊 Model Statistics:
   Total parameters: 44,777,984
   Trainable parameters: 44,777,984
   Model size: ~179.1 MB (fp32)


In [ ]:
# Cell #6: Test Forward Pass

# Create sample input (random token IDs)
batch_size = 2
seq_len = 10
sample_input = torch.randint(0, config.vocab_size, (batch_size, seq_len)).to(device)

print("🧪 Testing forward pass...")
print(f"Input shape: {sample_input.shape}")
print(f"Sample token IDs: {sample_input[0][:5].tolist()}")

# Forward pass
with torch.no_grad():  # No gradients needed for testing
    logits = model(sample_input)

print(f"\n✅ Forward pass successful!")
print(f"Output logits shape: {logits.shape}")
print(f"Expected shape: [batch_size={batch_size}, seq_len={seq_len}, vocab_size={config.vocab_size}]")

# Check logits for first token
print(f"\n📊 Logits for first token in batch:")
print(f"   Min: {logits[0, 0].min().item():.4f}")
print(f"   Max: {logits[0, 0].max().item():.4f}")
print(f"   Mean: {logits[0, 0].mean().item():.4f}")

print("\n💡 The model can now predict next tokens!")
print("   Each position has a distribution over the entire vocabulary.")

🧪 Testing forward pass...
Input shape: torch.Size([2, 10])
Sample token IDs: [45454, 41464, 22768, 20638, 17567]

✅ Forward pass successful!
Output logits shape: torch.Size([2, 10, 50257])
Expected shape: [batch_size=2, seq_len=10, vocab_size=50257]

📊 Logits for first token in batch:
   Min: -95.0797
   Max: 208.2510
   Mean: -0.1063

💡 The model can now predict next tokens!
   Each position has a distribution over the entire vocabulary.


In [ ]:
# Cell #7: Test Gradient Flow

# Create sample input with gradient tracking
sample_input = torch.randint(0, config.vocab_size, (batch_size, seq_len)).to(device)
model.train()  # Set to training mode

# Forward pass
logits = model(sample_input)

# Create dummy loss (mean of logits)
loss = logits.mean()

# Backward pass
loss.backward()

# Check gradients in first and last transformer blocks
first_block = model.blocks[0]
last_block = model.blocks[-1]

first_grad = first_block.attention.W_q.weight.grad
last_grad = last_block.attention.W_q.weight.grad

print("✅ Gradient Flow Test:")
print(f"\nFirst transformer block (layer 0):")
print(f"   Gradient exists: {first_grad is not None}")
if first_grad is not None:
    print(f"   Gradient norm: {first_grad.norm().item():.6f}")

print(f"\nLast transformer block (layer {config.n_layers-1}):")
print(f"   Gradient exists: {last_grad is not None}")
if last_grad is not None:
    print(f"   Gradient norm: {last_grad.norm().item():.6f}")

print("\n💡 Gradients flow through all 6 layers!")
print("   Thanks to residual connections, we can train deep networks.")

# Clear gradients
model.zero_grad()

✅ Gradient Flow Test:

First transformer block (layer 0):
   Gradient exists: True
   Gradient norm: 0.107632

Last transformer block (layer 5):
   Gradient exists: True
   Gradient norm: 0.039579

💡 Gradients flow through all 6 layers!
   Thanks to residual connections, we can train deep networks.


In [ ]:
# Cell #8: Day 6-7 Summary

print("=" * 60)
print("DAY 6-7 COMPLETE: GPT MODEL FROM SCRATCH ✓")
print("=" * 60)

print("\n🏗️ Complete Architecture Built:")
print(f"   1. Token Embedding: {config.vocab_size:,} vocab → {config.d_model} dim")
print(f"   2. Positional Embedding: Learned, max_len={config.max_seq_len}")
print(f"   3. Transformer Stack: {config.n_layers} blocks")
print(f"      • Multi-head attention ({config.num_heads} heads)")
print(f"      • Feed-forward network (d_ff={config.d_ff})")
print(f"      • Layer normalization + residuals")
print(f"   4. Causal Masking: Autoregressive attention")
print(f"   5. LM Head: {config.d_model} → {config.vocab_size:,} (weight-tied)")

print(f"\n📊 Model Statistics:")
print(f"   • Total parameters: {total_params:,}")
print(f"   • Model size: ~{total_params * 4 / 1e6:.1f} MB")
print(f"   • Can process sequences up to {config.max_seq_len} tokens")

print("\n✅ Key Features:")
print("   • End-to-end pipeline: token IDs → logits")
print("   • Causal attention for autoregressive generation")
print("   • Weight tying between embeddings and output")
print("   • Verified gradient flow through all layers")

print("\n🎯 What This Model Can Do:")
print("   • Generate text autoregressively")
print("   • Be trained on any text corpus")
print("   • Predict next tokens given context")

print("\n" + "=" * 60)
print("NEXT: Day 8-9 will cover Training & Optimization!")
print("=" * 60)

DAY 6-7 COMPLETE: GPT MODEL FROM SCRATCH ✓

🏗️ Complete Architecture Built:
   1. Token Embedding: 50,257 vocab → 512 dim
   2. Positional Embedding: Learned, max_len=256
   3. Transformer Stack: 6 blocks
      • Multi-head attention (8 heads)
      • Feed-forward network (d_ff=2048)
      • Layer normalization + residuals
   4. Causal Masking: Autoregressive attention
   5. LM Head: 512 → 50,257 (weight-tied)

📊 Model Statistics:
   • Total parameters: 44,777,984
   • Model size: ~179.1 MB
   • Can process sequences up to 256 tokens

✅ Key Features:
   • End-to-end pipeline: token IDs → logits
   • Causal attention for autoregressive generation
   • Weight tying between embeddings and output
   • Verified gradient flow through all layers

🎯 What This Model Can Do:
   • Generate text autoregressively
   • Be trained on any text corpus
   • Predict next tokens given context

NEXT: Day 8-9 will cover Training & Optimization!
